In [16]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import re
from pathlib import Path

In [17]:
BANK_NAME = "KB Kookmin Bank"
BANK_CODE = "KB"
PAGE_CODE = "C101408"
URL = f"https://obank.kbstar.com/quics?page={PAGE_CODE}&cc=b102292:b102292"

SAVE_DIR = Path("./outputs_kb_fx")
SAVE_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://obank.kbstar.com",
    "Referer": f"https://obank.kbstar.com/quics?page={PAGE_CODE}",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
}

In [18]:
def generate_quarter_end_dates(start_year=2004, end_year=2019):
    dates = []
    for y in range(start_year, end_year + 1):
        dates.extend([
            f"{y}0331",
            f"{y}0630",
            f"{y}0930",
            f"{y}1231",
        ])
    return dates

target_dates = generate_quarter_end_dates(2004, 2019)
target_dates[:8], target_dates[-4:]

(['20040331',
  '20040630',
  '20040930',
  '20041231',
  '20050331',
  '20050630',
  '20050930',
  '20051231'],
 ['20190331', '20190630', '20190930', '20191231'])

In [19]:
def fetch_kb_html(query_date: str, session: requests.Session) -> str:
    """
    query_date: YYYYMMDD
    """
    payload = {
        "STEP": "0",
        "조회년월일": query_date,
        "strFocusBtn": "",
        "tabNumber": "0",
        "MMDA고객구분": "",
        "se_inqueryYYYY": query_date[:4],
        "se_inqueryMM": query_date[4:6],
        "se_inqueryDD": query_date[6:8],
        "MMDA통화코드": "USD",  # 페이지 기본값. MMDA 탭용
    }

    r = session.post(URL, data=payload, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

In [20]:
def extract_observed_date(html: str, fallback_date: str) -> str:
    """
    HTML 안의 '조회기준일 : 2023.01.06' 같은 문자열을 읽어 observed_date 추출
    실패하면 fallback_date 사용
    """
    m = re.search(r"조회기준일\s*:\s*(\d{4})\.(\d{2})\.(\d{2})", html)
    if m:
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    return f"{fallback_date[:4]}-{fallback_date[4:6]}-{fallback_date[6:8]}"

In [21]:
TABLE_META = {
    "viewType_1_0": {"product_group": "정기예금", "residency": "resident"},
    "viewType_2_0": {"product_group": "정기예금", "residency": "nonresident"},
    "viewType_3_0": {"product_group": "보통예금", "residency": "mixed"},
    "viewType_4_0": {"product_group": "KB수출입기업우대외화통장", "residency": "mixed"},
    "viewType_5_0": {"product_group": "KB국민UP외화정기예금", "residency": "resident"},
    "viewType_6_0": {"product_group": "KB외화MMDA(개인)", "residency": "mixed"},
    "viewType_7_0": {"product_group": "외화적금", "residency": "resident"},
    # hidden expanded tables
    "targetTable1": {"product_group": "정기예금_확장", "residency": "resident"},
    "targetTable2": {"product_group": "정기예금_확장", "residency": "nonresident"},
    "targetTable3": {"product_group": "KB국민UP외화정기예금_확장", "residency": "resident"},
    "targetTable4": {"product_group": "외화적금_확장", "residency": "resident"},
}

In [22]:
def clean_currency(text: str) -> str:
    """
    'USD(미국 달러)' -> 'USD'
    """
    if pd.isna(text):
        return None
    text = str(text).strip()
    m = re.match(r"([A-Z]{3})", text)
    return m.group(1) if m else text

def clean_rate(x):
    if pd.isna(x):
        return None
    x = str(x).strip().replace(",", "")
    if x in ["", "-", "nan", "None"]:
        return None
    try:
        return float(x)
    except:
        return None

In [23]:
from io import StringIO

def parse_single_table(table_tag, table_id, target_date, observed_date):
    meta = TABLE_META.get(table_id, {"product_group": table_id, "residency": None})

    # 🔥 핵심 수정 (여기!)
    dfs = pd.read_html(StringIO(str(table_tag)))

    if len(dfs) == 0:
        return pd.DataFrame()

    df = dfs[0].copy()
    df.columns = [str(c).strip() for c in df.columns]

    rows = []

    if "통화" in df.columns:
        for _, r in df.iterrows():
            currency = clean_currency(r["통화"])

            for col in df.columns:
                if col == "통화":
                    continue

                rate = clean_rate(r[col])

                rows.append({
                    "bank": BANK_NAME,
                    "bank_code": BANK_CODE,
                    "target_date": f"{target_date[:4]}-{target_date[4:6]}-{target_date[6:8]}",
                    "observed_date": observed_date,
                    "product_group": meta["product_group"],
                    "residency": meta["residency"],
                    "currency": currency,
                    "maturity": col,
                    "rate": rate,
                    "unit": "annual %",
                    "table_id": table_id,
                })

    return pd.DataFrame(rows)

In [24]:
def parse_all_tables(html: str, target_date: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    observed_date = extract_observed_date(html, target_date)

    all_parts = []

    for table_id in TABLE_META.keys():
        tag = soup.find("table", {"id": table_id})
        if tag is not None:
            part = parse_single_table(tag, table_id, target_date, observed_date)
            if not part.empty:
                all_parts.append(part)

    if not all_parts:
        return pd.DataFrame()

    out = pd.concat(all_parts, ignore_index=True)

    # 불필요한 전부 NaN 줄 제거
    out = out.dropna(subset=["currency", "maturity"], how="any")
    return out

In [ ]:
!pip install html5lib
!pip install beautifulsoup4


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
session = requests.Session()

test_date = "20230106"
html = fetch_kb_html(test_date, session)
df_test = parse_all_tables(html, test_date)

print(df_test.head(20))
print(df_test["product_group"].value_counts(dropna=False))
print(df_test["observed_date"].drop_duplicates().tolist())

FeatureNotFound: Couldn't find a tree builder with the features you requested: html5lib. Do you need to install a parser library?

In [ ]:
probe_dates = ["20230106", "20190106", "20160106", "20100106", "20040331"]

probe_results = []

for d in probe_dates:
    try:
        html = fetch_kb_html(d, session)
        obs = extract_observed_date(html, d)
        has_error = ("고객님 죄송합니다" in html) or ("외화예금 금리조회는 최근" in html)
        probe_results.append({
            "query_date": d,
            "observed_date": obs,
            "html_length": len(html),
            "blocked_or_error": has_error
        })
        time.sleep(0.5)
    except Exception as e:
        probe_results.append({
            "query_date": d,
            "observed_date": None,
            "html_length": None,
            "blocked_or_error": True,
            "error": str(e)
        })

pd.DataFrame(probe_results)

In [ ]:
session = requests.Session()

all_data = []
fail_log = []

for i, d in enumerate(target_dates, start=1):
    try:
        html = fetch_kb_html(d, session)

        # 서버 에러/차단 체크
        if "고객님 죄송합니다" in html:
            fail_log.append({"query_date": d, "reason": "server_error_message"})
            continue

        parsed = parse_all_tables(html, d)

        if parsed.empty:
            fail_log.append({"query_date": d, "reason": "no_table_parsed"})
        else:
            all_data.append(parsed)

        if i % 5 == 0:
            print(f"{i}/{len(target_dates)} done: {d}")

        time.sleep(0.7)

    except Exception as e:
        fail_log.append({"query_date": d, "reason": str(e)})
        time.sleep(1.0)

kb_all = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
kb_fail = pd.DataFrame(fail_log)

print("rows:", len(kb_all))
print("fails:", len(kb_fail))
kb_all.head()

In [ ]:
# 보기 좋게 정리
kb_all["target_date"] = pd.to_datetime(kb_all["target_date"])
kb_all["observed_date"] = pd.to_datetime(kb_all["observed_date"], errors="coerce")

# rate가 모두 None인 경우 제거
kb_all_clean = kb_all.dropna(subset=["rate"]).copy()

# 핵심 통화 플래그
kb_all_clean["is_key_currency"] = kb_all_clean["currency"].isin(["USD", "JPY", "CNY", "EUR", "GBP"])

# 교수님이 중요하게 보실 요약
summary = (
    kb_all_clean.groupby(["product_group", "residency", "currency"])["rate"]
    .agg(["count", "min", "max"])
    .reset_index()
)

summary.head(30)

In [ ]:
output_path = SAVE_DIR / "kb_fx_all_maturities_2004_2019.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    kb_all.to_excel(writer, sheet_name="raw_all", index=False)
    kb_all_clean.to_excel(writer, sheet_name="clean_nonmissing", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    kb_fail.to_excel(writer, sheet_name="fail_log", index=False)

print("saved to:", output_path)

In [ ]:
key_panel = kb_all_clean[kb_all_clean["currency"].isin(["USD", "JPY", "CNY", "EUR", "GBP"])].copy()

key_output_path = SAVE_DIR / "kb_fx_keycurrencies_2004_2019.xlsx"
key_panel.to_excel(key_output_path, index=False)

print("saved key file:", key_output_path)